# Seeded Run 1 Retrain — 2-Class Task (Overfitting Fixes)

Retrains the five seeded pairs with regularization to address overfitting seen in the original run.

**Original run issues:** train→91-94%, test→73-85% — 8-20pp overfitting gap, high test volatility.

**Fixes applied:**
| Fix | Original | Retrain | Why |
|-----|----------|---------|-----|
| Weight decay | 0.0 | **5e-4** | Stronger L2 regularization |
| Learning rate | 4e-3 | **2e-3** | Smoother gradients, less oscillation |
| LR scheduler | None | **ReduceLROnPlateau(patience=5)** | Cut LR when test plateaus |
| Epochs | 25 | **25** | Same — convergence was fast enough |
| Batch size | 256 | **256** | Keep same |

Uses a separate checkpoint directory (`seeded_run_1_retrain`) to avoid conflicts.

In [1]:
import sys
sys.path.insert(0, r"C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files")

from pathlib import Path
import importlib

import pandas as pd
from IPython.display import display

import seeded_runs_common as seeded_runs_common

seeded_runs_common = importlib.reload(seeded_runs_common)

CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_sampling_preview_rows = seeded_runs_common.build_sampling_preview_rows
build_seeded_pair = seeded_runs_common.build_seeded_pair
expected_2class_odd_even_map = seeded_runs_common.expected_2class_odd_even_map
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

RUN_LABEL = "seeded_run_1_retrain"          # separate from original run
TASK_KEY = "2class"
MEM_DISTRIBUTION_FAMILY = "lognormal"
MASTER_SEEDS = [101, 202, 210, 340, 440]

RUN_DIR = CHECKPOINT_ROOT / RUN_LABEL / f"{TASK_KEY}_{MEM_DISTRIBUTION_FAMILY}"
RESULT_STEM = RUN_DIR / f"{RUN_LABEL}_checkpoint_summary"
PAIR_STEM = RUN_DIR / f"{RUN_LABEL}_pair_summary"
PAIRED_ACC_CSV = RUN_DIR / f"{RUN_LABEL}_paired_accuracy.csv"

TASK_DEF = TASKS[TASK_KEY]
EXPECTED_LABEL_MAP = expected_2class_odd_even_map()

print(f"Device: {DEVICE}")
print(f"Run directory: {RUN_DIR}")
print(f"Master seeds: {MASTER_SEEDS}")
print(f"Task name: {TASK_DEF['task_name']}")
print(f"Task outputs: {TASK_DEF['nb_outputs']}")

Device: cuda
Run directory: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1_retrain\2class_lognormal
Master seeds: [101, 202, 210, 340, 440]
Task name: binary_parity
Task outputs: 2


In [2]:
assert TASK_KEY == "2class"
assert TASK_DEF["nb_outputs"] == 2
assert TASK_DEF["task_name"] == "binary_parity"
assert TASK_DEF["task_label_map"] == EXPECTED_LABEL_MAP

preview = build_seeded_pair(
    master_seed=MASTER_SEEDS[0],
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
preview_meta = preview["metadata"]

assert preview["hom_prms"]["nb_outputs"] == 2
assert preview["hetero_prms"]["nb_outputs"] == 2
assert preview_meta["linear_sync_verified"]
assert preview_meta["hom_hidden_tau_unique"] == 1
assert preview_meta["hetero_hidden_tau_unique"] > 1

sampling_rows = build_sampling_preview_rows(
    MASTER_SEEDS,
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
sampling_df = pd.DataFrame(sampling_rows).sort_values("pair_seed").reset_index(drop=True)
assert not sampling_df["sample_matches_previous"].any()

display(sampling_df)
print("2-class odd/even task mapping verified.")
print("Homogeneous anchor and heterogeneous sampled definitions verified.")
print("Log-normal hidden tau_m sampling varies across the configured seeds.")

,pair_seed,task_key,task_name,mem_distribution_family,linear_sync_verified,hetero_hidden_tau_unique,hom_hidden_tau_unique,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,sample_matches_previous
0,101,2class,binary_parity,lognormal,True,30,1,24.687252,3.410472,99.749887,37.499115,30.195999,False
1,202,2class,binary_parity,lognormal,True,28,1,29.308363,6.953492,99.749887,39.884979,30.480470,False
2,210,2class,binary_parity,lognormal,True,31,1,22.405558,3.855315,99.749887,34.397371,29.713145,False
3,340,2class,binary_parity,lognormal,True,31,1,23.273864,4.656051,99.749887,35.008043,30.389582,False
4,440,2class,binary_parity,lognormal,True,28,1,28.874491,5.547958,99.749887,41.448743,31.915950,False


2-class odd/even task mapping verified.
Homogeneous anchor and heterogeneous sampled definitions verified.
Log-normal hidden tau_m sampling varies across the configured seeds.


In [3]:
train_cache, test_cache = load_default_caches()

Pre-loading SHD training data into RAM...
  SHDCache: 8156 samples loaded from shd_train.h5
Pre-loading SHD test data into RAM...
  SHDCache: 2264 samples loaded from shd_test.h5


In [4]:
# ── 2-Class Retrain: Overfitting Fixes + Speed ─────────────────────
#   1. batch_size 256→512: halves iterations/epoch (~1.8× faster)
#   2. weight_decay 0→5e-4: L2 regularization to close train-test gap
#   3. lr 4e-3→2e-3: smoother gradients, reduce test volatility
#   4. LR scheduler patience=5: cut LR when test loss plateaus
#   5. cuDNN benchmark: auto-tune kernels

import torch
import importlib

# Reload common module
seeded_runs_common = importlib.reload(seeded_runs_common)

# Re-bind names
CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_seeded_pair = seeded_runs_common.build_seeded_pair
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

TASK_DEF = TASKS[TASK_KEY]

# ── Speed settings ──
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

seeded_runs_common.BASE_PRMS["batch_size"] = 512          # was 256: halves iterations
seeded_runs_common.BASE_PRMS["compile_model"] = False     # Triton not available

# ── Overfitting fixes ──
seeded_runs_common.BASE_PRMS["weight_decay"] = 5e-4       # was 0.0: L2 regularization
seeded_runs_common.BASE_PRMS["lr"] = 2e-3                  # was 4e-3: smoother gradients
seeded_runs_common.BASE_PRMS["lr_ab"] = 2e-3               # was 4e-3: match
seeded_runs_common.BASE_PRMS["lr_patience"] = 5            # scheduler: wait 5 epochs
seeded_runs_common.BASE_PRMS["lr_factor"] = 0.5            # halve LR on plateau
seeded_runs_common.BASE_PRMS["lr_min"] = 5e-5              # floor

print("2-class retrain: speed + overfitting fixes applied:")
print(f"  batch_size     = {seeded_runs_common.BASE_PRMS['batch_size']}  (was 256)")
print(f"  weight_decay   = {seeded_runs_common.BASE_PRMS['weight_decay']}")
print(f"  lr / lr_ab     = {seeded_runs_common.BASE_PRMS['lr']:.0e}  (was 4e-3)")
print(f"  LR scheduler   = ReduceLROnPlateau(patience=5, factor=0.5)")
print(f"  cuDNN benchmark = {torch.backends.cudnn.benchmark if torch.cuda.is_available() else 'N/A'}")
print(f"  Est. time/seed: ~25 min (25 epochs × ~1 min with batch 512)")


2-class retrain: speed + overfitting fixes applied:
  batch_size     = 512  (was 256)
  weight_decay   = 0.0005
  lr / lr_ab     = 2e-03  (was 4e-3)
  LR scheduler   = ReduceLROnPlateau(patience=5, factor=0.5)
  cuDNN benchmark = True
  Est. time/seed: ~25 min (25 epochs × ~1 min with batch 512)


In [5]:
CHECKPOINT_KEY_FIELDS = ["pair_seed", "pair_role"]
PAIR_KEY_FIELDS = ["pair_seed"]

def show_run_status():
    checkpoint_rows = read_manifest_rows(RESULT_STEM)
    pair_rows = read_manifest_rows(PAIR_STEM)

    checkpoint_df = pd.DataFrame(checkpoint_rows)
    pair_df = pd.DataFrame(pair_rows)

    if not checkpoint_df.empty:
        checkpoint_df = checkpoint_df.sort_values(["pair_seed", "pair_role"]).reset_index(drop=True)
    if not pair_df.empty:
        pair_df = pair_df.sort_values("pair_seed").reset_index(drop=True)

    paired_acc_df = pd.DataFrame()
    if not checkpoint_df.empty:
        paired_acc_df = (
            checkpoint_df.pivot(index="pair_seed", columns="pair_role", values="final_test_acc")
            .reset_index()
            .sort_values("pair_seed")
            .reset_index(drop=True)
        )
        if {"heterogeneous_sampled", "homogeneous_anchor"}.issubset(paired_acc_df.columns):
            paired_acc_df["hetero_minus_hom"] = (
                paired_acc_df["heterogeneous_sampled"] - paired_acc_df["homogeneous_anchor"]
            )
            paired_acc_df.to_csv(PAIRED_ACC_CSV, index=False)

    return pair_df, checkpoint_df, paired_acc_df

def train_one_seed(master_seed):
    run_rows, pair_meta = run_pair_training(
        master_seed=master_seed,
        train_cache=train_cache,
        test_cache=test_cache,
        task_key=TASK_KEY,
        mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
        run_label=RUN_LABEL,
        skip_existing=True,
    )

    checkpoint_rows = upsert_rows(
        read_manifest_rows(RESULT_STEM),
        run_rows,
        CHECKPOINT_KEY_FIELDS,
    )
    pair_rows = upsert_rows(
        read_manifest_rows(PAIR_STEM),
        [build_pair_summary_row(pair_meta)],
        PAIR_KEY_FIELDS,
    )

    write_manifest_rows(checkpoint_rows, RESULT_STEM)
    write_manifest_rows(pair_rows, PAIR_STEM)

    pair_df, checkpoint_df, paired_acc_df = show_run_status()
    seed_df = checkpoint_df[checkpoint_df["pair_seed"] == master_seed].reset_index(drop=True)
    return seed_df, pair_df, checkpoint_df, paired_acc_df

print("Run helpers ready.")
print("Execute one seed cell at a time. Finished seeds are reused from saved checkpoints.")
print("Chance level for 2-class task: 50.0%")

Run helpers ready.
Execute one seed cell at a time. Finished seeds are reused from saved checkpoints.
Chance level for 2-class task: 50.0%


In [6]:
# Train or reuse seed pair 101
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(101)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 101].reset_index(drop=True))


Seed 101 :: training homogeneous_anchor...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.529  test_acc=0.519  (1.6 min)
  epoch=002  train_acc=0.521  test_acc=0.515  (1.4 min)
  epoch=003  train_acc=0.536  test_acc=0.532  (1.4 min)
  epoch=004  train_acc=0.542  test_acc=0.542  (1.4 min)
  epoch=005  train_acc=0.547  test_acc=0.526  (1.4 min)
  epoch=006  train_acc=0.546  test_acc=0.542  (1.4 min)
  epoch=007  train_acc=0.551  test_acc=0.532  (1.4 min)
  epoch=008  train_acc=0.553  test_acc=0.542  (1.4 min)
  epoch=009  train_acc=0.554  test_acc=0.542  (1.4 min)
  epoch=010  train_acc=0.583  test_acc=0.639  (1.4 min)
  epoch=011  train_acc=0.671  test_acc=0.698  (1.4 min)
  epoch=012  train_acc=0.682  test_acc=0.707  (1.4 min)
  epoch=013  train_acc=0.690  test_acc=0.697  (1.4 min)
  epoch=014  train_acc=0.712  test_acc=0.657  (1.4 min)
  epoch=015  train_acc=0.700  test_acc=0.715  (1.4 min)
  epoch=016  train_acc=0.699  test_acc=0.648  (1.4 min)
  epoch=017  train_acc=0.728  test_acc=0.651  (1.4 min)
  epoch=018  train_acc=0.727  test_acc=0.668  (1

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,101,heterogeneous_sampled,0.713339,0.537695,2097.978802
1,101,homogeneous_anchor,0.666078,0.565709,2085.420809


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,101,2class,binary_parity,lognormal,10.0,24.687252,3.410472,99.749887,37.499115,30.195999,30,1,True,6


In [7]:
# Train or reuse seed pair 202
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(202)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 202].reset_index(drop=True))


Seed 202 :: training homogeneous_anchor...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.496  test_acc=0.498  (1.8 min)
  epoch=002  train_acc=0.501  test_acc=0.514  (2.0 min)
  epoch=003  train_acc=0.508  test_acc=0.516  (2.0 min)
  epoch=004  train_acc=0.520  test_acc=0.530  (2.0 min)
  epoch=005  train_acc=0.570  test_acc=0.655  (2.1 min)
  epoch=006  train_acc=0.604  test_acc=0.583  (2.0 min)
  epoch=007  train_acc=0.639  test_acc=0.682  (1.9 min)
  epoch=008  train_acc=0.655  test_acc=0.668  (1.9 min)
  epoch=009  train_acc=0.654  test_acc=0.667  (2.0 min)
  epoch=010  train_acc=0.668  test_acc=0.676  (2.1 min)
  epoch=011  train_acc=0.661  test_acc=0.695  (2.1 min)
  epoch=012  train_acc=0.675  test_acc=0.686  (2.2 min)
  epoch=013  train_acc=0.674  test_acc=0.679  (1.7 min)
  epoch=014  train_acc=0.688  test_acc=0.696  (1.7 min)
  epoch=015  train_acc=0.684  test_acc=0.609  (1.8 min)
  epoch=016  train_acc=0.679  test_acc=0.686  (1.7 min)
  epoch=017  train_acc=0.693  test_acc=0.689  (1.7 min)
  epoch=018  train_acc=0.687  test_acc=0.694  (1

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,202,heterogeneous_sampled,0.720848,0.541853,2605.405963
1,202,homogeneous_anchor,0.670053,0.583959,2794.536900


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,202,2class,binary_parity,lognormal,10.0,29.308363,6.953492,99.749887,39.884979,30.48047,28,1,True,6


In [8]:
# Train or reuse seed pair 210
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(210)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 210].reset_index(drop=True))


Seed 210 :: training homogeneous_anchor...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.522  test_acc=0.522  (1.8 min)
  epoch=002  train_acc=0.527  test_acc=0.509  (1.8 min)
  epoch=003  train_acc=0.578  test_acc=0.649  (1.8 min)
  epoch=004  train_acc=0.660  test_acc=0.671  (1.7 min)
  epoch=005  train_acc=0.667  test_acc=0.683  (1.7 min)
  epoch=006  train_acc=0.671  test_acc=0.655  (1.7 min)
  epoch=007  train_acc=0.687  test_acc=0.675  (1.6 min)
  epoch=008  train_acc=0.683  test_acc=0.617  (1.7 min)
  epoch=009  train_acc=0.683  test_acc=0.697  (1.7 min)
  epoch=010  train_acc=0.677  test_acc=0.617  (1.7 min)
  epoch=011  train_acc=0.694  test_acc=0.629  (1.7 min)
  epoch=012  train_acc=0.699  test_acc=0.700  (1.7 min)
  epoch=013  train_acc=0.708  test_acc=0.719  (1.7 min)
  epoch=014  train_acc=0.703  test_acc=0.667  (1.7 min)
  epoch=015  train_acc=0.713  test_acc=0.648  (1.7 min)
  epoch=016  train_acc=0.706  test_acc=0.715  (1.7 min)
  epoch=017  train_acc=0.701  test_acc=0.635  (1.7 min)
  epoch=018  train_acc=0.710  test_acc=0.666  (1

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,210,heterogeneous_sampled,0.720848,0.556108,2537.074986
1,210,homogeneous_anchor,0.707155,0.549172,2558.249054


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,210,2class,binary_parity,lognormal,10.0,22.405558,3.855315,99.749887,34.397371,29.713145,31,1,True,6


In [9]:
# Train or reuse seed pair 340
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(340)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 340].reset_index(drop=True))


Seed 340 :: training homogeneous_anchor...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.555  test_acc=0.551  (1.7 min)
  epoch=002  train_acc=0.566  test_acc=0.551  (1.7 min)
  epoch=003  train_acc=0.568  test_acc=0.530  (1.7 min)
  epoch=004  train_acc=0.558  test_acc=0.549  (1.7 min)
  epoch=005  train_acc=0.561  test_acc=0.559  (1.7 min)
  epoch=006  train_acc=0.558  test_acc=0.555  (1.7 min)
  epoch=007  train_acc=0.562  test_acc=0.551  (1.7 min)
  epoch=008  train_acc=0.567  test_acc=0.561  (1.7 min)
  epoch=009  train_acc=0.599  test_acc=0.557  (1.7 min)
  epoch=010  train_acc=0.636  test_acc=0.614  (1.7 min)
  epoch=011  train_acc=0.678  test_acc=0.693  (1.7 min)
  epoch=012  train_acc=0.677  test_acc=0.700  (1.7 min)
  epoch=013  train_acc=0.682  test_acc=0.664  (1.7 min)
  epoch=014  train_acc=0.707  test_acc=0.645  (1.8 min)
  epoch=015  train_acc=0.715  test_acc=0.648  (1.8 min)
  epoch=016  train_acc=0.719  test_acc=0.644  (1.8 min)
  epoch=017  train_acc=0.719  test_acc=0.723  (1.8 min)
  epoch=018  train_acc=0.723  test_acc=0.670  (1

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,340,heterogeneous_sampled,0.717756,0.531122,2585.804215
1,340,homogeneous_anchor,0.687721,0.552367,2586.959446


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,340,2class,binary_parity,lognormal,10.0,23.273864,4.656051,99.749887,35.008043,30.389582,31,1,True,6


In [10]:
# Train or reuse seed pair 440
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(440)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 440].reset_index(drop=True))


Seed 440 :: training homogeneous_anchor...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.521  test_acc=0.554  (1.7 min)
  epoch=002  train_acc=0.538  test_acc=0.567  (1.7 min)
  epoch=003  train_acc=0.562  test_acc=0.606  (1.7 min)
  epoch=004  train_acc=0.597  test_acc=0.559  (1.7 min)
  epoch=005  train_acc=0.618  test_acc=0.644  (1.7 min)
  epoch=006  train_acc=0.638  test_acc=0.646  (1.8 min)
  epoch=007  train_acc=0.646  test_acc=0.635  (1.5 min)
  epoch=008  train_acc=0.645  test_acc=0.634  (1.3 min)
  epoch=009  train_acc=0.658  test_acc=0.648  (1.7 min)
  epoch=010  train_acc=0.658  test_acc=0.641  (1.7 min)
  epoch=011  train_acc=0.664  test_acc=0.654  (1.7 min)
  epoch=012  train_acc=0.671  test_acc=0.648  (1.9 min)
  epoch=013  train_acc=0.675  test_acc=0.650  (1.9 min)
  epoch=014  train_acc=0.677  test_acc=0.656  (1.9 min)
  epoch=015  train_acc=0.675  test_acc=0.675  (1.9 min)
  epoch=016  train_acc=0.684  test_acc=0.682  (1.9 min)
  epoch=017  train_acc=0.688  test_acc=0.688  (1.9 min)
  epoch=018  train_acc=0.690  test_acc=0.678  (1

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,440,heterogeneous_sampled,0.666961,0.572678,2556.204337
1,440,homogeneous_anchor,0.688163,0.560829,2589.032457


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,440,2class,binary_parity,lognormal,10.0,28.874491,5.547958,99.749887,41.448743,31.91595,28,1,True,6


In [11]:
# Final status summary
pair_df, checkpoint_df, paired_acc_df = show_run_status()

if pair_df.empty:
    print("No saved seed summaries yet.")
else:
    display(pair_df)

if checkpoint_df.empty:
    print("No saved checkpoint summaries yet.")
else:
    display(checkpoint_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss"]])

if not paired_acc_df.empty:
    display(paired_acc_df)
    print(f"Saved paired accuracy view to: {PAIRED_ACC_CSV}")
    
    mean_delta = paired_acc_df["hetero_minus_hom"].mean()
    print(f"\nMean hetero − homo Δacc: {mean_delta*100:+.2f} pp")
    
    # Compare to original run
    print("\nComparison to original run (seeded_run_1):")
    print("  Original mean Δ: +2.69 pp (run 1), -2.44 pp (run 2)")
    if mean_delta >= 0.03:
        print("  ✓ Retrain shows ≥ 3pp heterogeneity advantage")
    elif mean_delta >= 0.02:
        print("  ~ Retrain shows ≥ 2pp heterogeneity advantage")
    else:
        print("  — No clear heterogeneity advantage")

,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,101,2class,binary_parity,lognormal,10.0,24.687252,3.410472,99.749887,37.499115,30.195999,30,1,True,6
1,202,2class,binary_parity,lognormal,10.0,29.308363,6.953492,99.749887,39.884979,30.480470,28,1,True,6
2,210,2class,binary_parity,lognormal,10.0,22.405558,3.855315,99.749887,34.397371,29.713145,31,1,True,6
3,340,2class,binary_parity,lognormal,10.0,23.273864,4.656051,99.749887,35.008043,30.389582,31,1,True,6
4,440,2class,binary_parity,lognormal,10.0,28.874491,5.547958,99.749887,41.448743,31.915950,28,1,True,6


,pair_seed,pair_role,final_test_acc,final_test_loss
0,101,heterogeneous_sampled,0.713339,0.537695
1,101,homogeneous_anchor,0.666078,0.565709
2,202,heterogeneous_sampled,0.720848,0.541853
3,202,homogeneous_anchor,0.670053,0.583959
4,210,heterogeneous_sampled,0.720848,0.556108
5,210,homogeneous_anchor,0.707155,0.549172
6,340,heterogeneous_sampled,0.717756,0.531122
7,340,homogeneous_anchor,0.687721,0.552367
8,440,heterogeneous_sampled,0.666961,0.572678
9,440,homogeneous_anchor,0.688163,0.560829


pair_role,pair_seed,heterogeneous_sampled,homogeneous_anchor,hetero_minus_hom
0,101,0.713339,0.666078,0.047261
1,202,0.720848,0.670053,0.050795
2,210,0.720848,0.707155,0.013693
3,340,0.717756,0.687721,0.030035
4,440,0.666961,0.688163,-0.021201


Saved paired accuracy view to: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1_retrain\2class_lognormal\seeded_run_1_retrain_paired_accuracy.csv

Mean hetero − homo Δacc: +2.41 pp

Comparison to original run (seeded_run_1):
  Original mean Δ: +2.69 pp (run 1), -2.44 pp (run 2)
  ~ Retrain shows ≥ 2pp heterogeneity advantage


---

## Statistical Significance Tests


In [12]:
import numpy as np
from scipy import stats

if not paired_acc_df.empty and len(paired_acc_df) >= 2:
    diffs = paired_acc_df["hetero_minus_hom"].values
    homo = paired_acc_df["homogeneous_anchor"].values
    hetero = paired_acc_df["heterogeneous_sampled"].values
    n = len(diffs)

    # ── Paired t-test (one-sided: hetero > homo) ──
    t_stat, p_ttest = stats.ttest_rel(hetero, homo, alternative="greater")
    mean_diff = np.mean(diffs)
    sd_diff = np.std(diffs, ddof=1)
    cohens_d = mean_diff / sd_diff if sd_diff > 0 else 0.0
    ci = stats.t.interval(0.95, df=n-1, loc=mean_diff, scale=sd_diff / np.sqrt(n))

    # ── Wilcoxon signed-rank (one-sided: hetero > homo) ──
    w_stat, p_wilcoxon = stats.wilcoxon(hetero, homo, alternative="greater")
    n_pos = int(np.sum(diffs > 0))
    n_neg = int(np.sum(diffs < 0))

    print("=" * 65)
    print("  2-Class Retrain — Statistical Significance")
    print("=" * 65)
    print(f"  Pairs (n):               {n}")
    print(f"  Hetero > Homo:            {n_pos}/{n} seeds")
    print(f"  Homo > Hetero:            {n_neg}/{n} seeds")
    print()
    print(f"  Mean Δacc:               {mean_diff*100:+.2f} pp")
    print(f"  SD of Δacc:              {sd_diff*100:.2f} pp")
    print(f"  95% CI for Δacc:         [{ci[0]*100:+.2f}, {ci[1]*100:+.2f}] pp")
    print()

    print(f"  ── Paired t-test (one-sided) ──")
    print(f"  t-statistic:             {t_stat:.4f}")
    print(f"  p-value:                 {p_ttest:.6f}")
    print(f"  Cohen's d:               {cohens_d:.3f}")
    print(f"  Significant @ α=0.05:    {p_ttest < 0.05}")
    if p_ttest < 0.001:      print(f"  ★★★ Highly significant (p < 0.001)")
    elif p_ttest < 0.01:     print(f"  ★★  Very significant (p < 0.01)")
    elif p_ttest < 0.05:     print(f"  ★   Significant (p < 0.05)")
    else:                    print(f"  —   Not significant (n={n} limits power)")

    print()
    print(f"  ── Wilcoxon signed-rank (one-sided) ──")
    print(f"  W-statistic:             {w_stat:.1f}")
    print(f"  p-value:                 {p_wilcoxon:.6f}")
    print(f"  Significant @ α=0.05:    {p_wilcoxon < 0.05}")
    if p_wilcoxon < 0.05:    print(f"  ★   Significant")
    else:                    print(f"  —   Not significant (min possible p with n=5 is 0.03125)")

    print()
    print(f"  ── Comparison ──")
    print(f"  Perez-Nieves benchmark:  +3–8 pp improvement on SHD")
    print(f"  Our observed mean Δ:     {mean_diff*100:+.2f} pp")
    if mean_diff >= 0.03:
        print(f"  ✓ Within Perez-Nieves range")
    elif mean_diff >= 0.02:
        print(f"  ~ Close to Perez-Nieves range (≥ 2pp)")
    else:
        print(f"  — Below Perez-Nieves benchmark")


  2-Class Retrain — Statistical Significance
  Pairs (n):               5
  Hetero > Homo:            4/5 seeds
  Homo > Hetero:            1/5 seeds

  Mean Δacc:               +2.41 pp
  SD of Δacc:              2.93 pp
  95% CI for Δacc:         [-1.23, +6.06] pp

  ── Paired t-test (one-sided) ──
  t-statistic:             1.8376
  p-value:                 0.069995
  Cohen's d:               0.822
  Significant @ α=0.05:    False
  —   Not significant (n=5 limits power)

  ── Wilcoxon signed-rank (one-sided) ──
  W-statistic:             13.0
  p-value:                 0.093750
  Significant @ α=0.05:    False
  —   Not significant (min possible p with n=5 is 0.03125)

  ── Comparison ──
  Perez-Nieves benchmark:  +3–8 pp improvement on SHD
  Our observed mean Δ:     +2.41 pp
  ~ Close to Perez-Nieves range (≥ 2pp)
